# 5. Durable HITL interrupt, resume, and idempotent writes

In [ ]:
print("Embedded dataset and deterministic offline lesson are ready.")

EDUCATIONAL — SELF-CONTAINED

A durable human-in-the-loop (HITL) gate pauses before a simulated write. This lesson uses the real LangGraph `SqliteSaver` checkpointer on a temporary on-disk SQLite database. We close the first graph/checkpointer, rebuild a new graph/checkpointer over that same database, and resume with the same `thread_id`. The write uses an idempotency key, so replay returns one logical receipt.

In [ ]:
import os
import tempfile
from typing import TypedDict

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


class HitlState(TypedDict, total=False):
    proposal: str
    idempotency_key: str
    decision: dict
    receipt: dict
    status: str


receipts: dict[str, dict] = {}

def write_once(key: str, action: str) -> dict:
    if key not in receipts:
        receipts[key] = {"receipt_id": "R-001", "action": action, "key": key}
    return receipts[key]


def build_graph(checkpointer):
    def review_node(state: HitlState) -> dict:
        decision = interrupt({"proposal": state["proposal"], "risk": "inventory hold"})
        return {"decision": decision, "status": "approved" if decision.get("approved") else "rejected"}

    def apply_node(state: HitlState) -> dict:
        if state["status"] != "approved":
            return {"receipt": {"status": "not written"}}
        return {"receipt": write_once(state["idempotency_key"], state["proposal"])}

    builder = StateGraph(HitlState)
    builder.add_node("review", review_node)
    builder.add_node("apply", apply_node)
    builder.add_edge(START, "review")
    builder.add_edge("review", "apply")
    builder.add_edge("apply", END)
    return builder.compile(checkpointer=checkpointer)


with tempfile.TemporaryDirectory() as temporary:
    database = os.path.join(temporary, "case-checkpoints.sqlite")
    config = {"configurable": {"thread_id": "case-H-1230-2026"}}
    with SqliteSaver.from_conn_string(database) as first_checkpointer:
        first_checkpointer.setup()
        first_graph = build_graph(first_checkpointer)
        paused = first_graph.invoke({"proposal": "hold SKU-1 at DC-West", "idempotency_key": "hold-case-1"}, config)
        print("paused interrupt ->", paused["__interrupt__"][0].value)
        assert paused["__interrupt__"][0].value["risk"] == "inventory hold"
    with SqliteSaver.from_conn_string(database) as rebuilt_checkpointer:
        rebuilt_checkpointer.setup()
        rebuilt_graph = build_graph(rebuilt_checkpointer)
        resumed = rebuilt_graph.invoke(Command(resume={"approved": True, "actor": "food-safety-manager"}), config)
        print("resumed after rebuild ->", resumed)
        first = resumed["receipt"]

second = write_once("hold-case-1", "hold SKU-1 at DC-West")
print("replayed receipt ->", second)
assert resumed["status"] == "approved"
assert first == second and first["receipt_id"] == "R-001"
print("ASSERTION PASSED: SQLite pause, rebuilt resume, same thread_id, and idempotent write replay worked")


In [ ]:
print("EDUCATIONAL — SELF-CONTAINED")
print("SQLite makes the checkpoint survive graph reconstruction; idempotency makes retry safe.")
assert list(receipts) == ["hold-case-1"]
print("ASSERTION PASSED: EDUCATIONAL — SELF-CONTAINED")